# Construção e Avaliação do Modelo de Marketing Mix

Neste notebook, construímos o modelo de Marketing Mix Modeling (MMM) passo a passo:
1. **Transformações de mídia** — adstock (carryover) e saturação de Hill.
2. **Ajuste do modelo** — regressão OLS com fallback restrito para coeficientes não-negativos.
3. **Diagnósticos** — R², MAPE, QQ-Plot, resíduos e VIF.
4. **Decomposição** — contribuição e ROI de cada canal.

In [1]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Adicionar raiz do projeto ao path para importar src/
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from src.data_generator import generate_marketing_data
from src.transformations import (
    DEFAULT_CHANNEL_PARAMS,
    apply_all_transformations,
    geometric_adstock,
    hill_saturation,
)
from src.model import CHANNELS, MarketingMixModel
from src.visualizations import (
    CHANNEL_COLORS,
    CHANNEL_LABELS,
    plot_adstock_effect,
    plot_model_diagnostics,
    plot_response_curves,
    plot_revenue_decomposition_waterfall,
    plot_roi_by_channel,
    plot_channel_contribution_over_time,
)

# Configurações visuais
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Arial", "DejaVu Sans"],
    "figure.dpi": 120,
    "figure.facecolor": "white",
})

# Carregar dados
caminho_csv = os.path.join("..", "data", "raw", "marketing_data.csv")
df = pd.read_csv(caminho_csv, parse_dates=["date"])
print(f"Dataset carregado: {df.shape[0]} semanas, {df.shape[1]} colunas")

Dataset carregado: 104 semanas, 17 colunas


## 1. Transformações de Mídia — Adstock e Saturação

Antes de alimentar o modelo, transformamos o investimento bruto de cada canal em duas etapas:

- **Adstock geométrico**: modela o *carryover* — parte do investimento de uma semana continua gerando impacto nas seguintes. Fórmula: `adstock_t = spend_t + decay × adstock_{t-1}`.
- **Saturação de Hill**: modela retornos decrescentes — dobrar o investimento **não** dobra o resultado. Fórmula: `saturated = 1 / (1 + (half_sat / x)^slope)`.

Cada canal tem parâmetros próprios calibrados para refletir seu comportamento real.

In [2]:
# Parâmetros de transformação por canal
params_df = pd.DataFrame(DEFAULT_CHANNEL_PARAMS).T
params_df.index = [CHANNEL_LABELS.get(c, c) for c in params_df.index]
params_df.columns = ["Decay (Adstock)", "Half-Saturation (R$)", "Slope (Hill)"]
print("Parâmetros de transformação por canal:")
params_df

Parâmetros de transformação por canal:


,Decay (Adstock),Half-Saturation (R$),Slope (Hill)
Meta Ads,0.55,18000.0,1.5
Google Ads,0.30,12000.0,1.8
LinkedIn Ads,0.60,8000.0,1.3
Email Marketing,0.20,3000.0,2.0
Conteúdo Orgânico,0.70,2500.0,2.5


In [3]:
# Aplicar transformações e visualizar o efeito do adstock em dois canais contrastantes
df_transformed = apply_all_transformations(df)

# Canal com alto carryover: Conteúdo Orgânico (decay=0.70)
fig_adstock_organic = plot_adstock_effect(df, df_transformed, "content_organic")
fig_adstock_organic.show()

# Canal com baixo carryover: Google Ads (decay=0.30)
fig_adstock_google = plot_adstock_effect(df, df_transformed, "google_ads")
fig_adstock_google.show()

### Interpretação do Adstock

Compare os dois canais:
- **Conteúdo Orgânico** (decay = 0.70): a linha adstockada fica muito acima e mais suave que a original — o efeito de um investimento se prolonga por várias semanas. Isso é esperado para conteúdo de branding.
- **Google Ads** (decay = 0.30): a diferença entre original e adstockado é menor — o efeito é mais imediato e se dissipa rapidamente. Típico de canais de performance.

Após o adstock, aplicamos a **saturação de Hill**, que comprime os valores para o intervalo [0, 1], representando o grau de saturação do canal.

## 2. Treinamento do Modelo

O modelo é uma **regressão linear** onde a variável dependente é a receita semanal e as features são:
- **Canais transformados** (5): investimento pós-adstock e pós-saturação.
- **Variáveis de contexto** (5): `seasonality_index`, `is_holiday`, `is_black_friday`, `competitor_spend_index` e `trend`.

Se algum coeficiente de canal sair negativo (o que não faz sentido em marketing), um fallback com **otimização restrita (L-BFGS-B)** é ativado para forçar não-negatividade.

In [4]:
# Treinar o modelo
mmm = MarketingMixModel()
mmm.fit(df)

# Resumo completo
mmm.summary()

                       MARKETING MIX MODEL — RESUMO                       
Método de ajuste      : OLS com bounds (L-BFGS-B)
Observações           : 104
Features              : 10

                           QUALIDADE DO AJUSTE                            
--------------------------------------------------------------------------
R²                    : 0.8833
R² ajustado           : 0.8707
MAPE                  : 3.16%
MAE                   : R$ 4,367.14
RMSE                  : R$ 5,266.69
Durbin-Watson         : 2.025  (~2 indica ausencia de autocorrelacao)

                               COEFICIENTES                               
--------------------------------------------------------------------------
  const                                          4,636.3093
  spend_meta_ads_transformed                    53,792.8122
  spend_google_ads_transformed                       0.0000
  spend_linkedin_ads_transformed                14,426.2218
  spend_email_marketing_transformed         

## 3. Diagnósticos do Modelo

Avaliamos a qualidade do ajuste com quatro visualizações clássicas e as métricas numéricas.

In [5]:
# Grid 2x2 de diagnósticos
fig_diag = plot_model_diagnostics(mmm)
fig_diag.show()

# Métricas numéricas
diag = mmm.get_model_diagnostics()
print("=" * 50)
print("MÉTRICAS DO MODELO")
print("=" * 50)
print(f"R²:              {diag['r2']:.4f}")
print(f"R² ajustado:     {diag['adj_r2']:.4f}")
print(f"MAPE:            {diag['mape']:.2f}%")
print(f"MAE:             R$ {diag['mae']:,.2f}")
print(f"RMSE:            R$ {diag['rmse']:,.2f}")
print(f"Durbin-Watson:   {diag['durbin_watson']:.3f}")
print(f"Ajuste restrito: {'Sim' if diag['is_constrained'] else 'Não'}")
print()

# VIF
print("MULTICOLINEARIDADE (VIF):")
print("-" * 50)
for feature, vif_val in diag["vif"].items():
    alerta = " ⚠ ALTO" if vif_val > 10 else ""
    nome_curto = feature.replace("spend_", "").replace("_transformed", "")
    print(f"  {nome_curto:<30s} {vif_val:>8.2f}{alerta}")

MÉTRICAS DO MODELO
R²:              0.8833
R² ajustado:     0.8707
MAPE:            3.16%
MAE:             R$ 4,367.14
RMSE:            R$ 5,266.69
Durbin-Watson:   2.025
Ajuste restrito: Sim

MULTICOLINEARIDADE (VIF):
--------------------------------------------------
  meta_ads                           6.13
  google_ads                         3.62
  linkedin_ads                       4.57
  email_marketing                    2.83
  content_organic                    5.08
  seasonality_index                  3.83
  is_holiday                         1.16
  is_black_friday                    1.14
  competitor_index                   3.43
  trend                              1.50


### Interpretação dos Diagnósticos

- **R² ~0.88**: o modelo explica ~88% da variação da receita — um ajuste forte para dados de marketing.
- **MAPE baixo**: o erro percentual médio indica que as previsões desviam pouco da receita real.
- **QQ-Plot**: os resíduos seguem razoavelmente a reta normal, sem caudas pesadas significativas.
- **Resíduos vs. Preditos**: sem padrão evidente — boa indicação de homocedasticidade.
- **Durbin-Watson ~2**: sem autocorrelação significativa nos resíduos.
- **VIF**: valores controlados indicam que a multicolinearidade está dentro de limites aceitáveis após as transformações.

## 4. Decomposição da Receita e ROI por Canal

Agora decompomos a receita total em baseline + contribuição de cada canal, e calculamos o retorno sobre investimento.

In [6]:
# Tabela de contribuição e ROI
contributions = mmm.get_channel_contributions()
contributions["canal_display"] = contributions["canal"].map(
    lambda c: CHANNEL_LABELS.get(c, c)
)

print("Contribuição e ROI por Canal:")
print("=" * 74)
for _, row in contributions.iterrows():
    roi_txt = f"{row['roi']:.2f}x" if pd.notna(row["roi"]) else "  —"
    print(
        f"  {CHANNEL_LABELS.get(row['canal'], row['canal']):<22s}"
        f"  Contrib: R$ {row['contribuicao_total']:>12,.0f}"
        f"  ({row['contribuicao_pct']:>5.1f}%)"
        f"  ROI: {roi_txt:>7s}"
    )

# Waterfall
fig_wf = plot_revenue_decomposition_waterfall(contributions)
fig_wf.show()

Contribuição e ROI por Canal:
  Meta Ads                Contrib: R$    1,900,491  ( 13.1%)  ROI:   3.45x
  Google Ads              Contrib: R$            0  (  0.0%)  ROI:   0.00x
  LinkedIn Ads            Contrib: R$      515,680  (  3.6%)  ROI:   2.50x
  Email Marketing         Contrib: R$      942,042  (  6.5%)  ROI:   6.84x
  Conteúdo Orgânico       Contrib: R$      862,612  (  6.0%)  ROI:  12.65x
  Baseline                Contrib: R$   10,261,473  ( 70.9%)  ROI:       —


In [7]:
# ROI por canal
fig_roi = plot_roi_by_channel(contributions)
fig_roi.show()

# Contribuição ao longo do tempo
fig_contrib = plot_channel_contribution_over_time(mmm, df)
fig_contrib.show()

## 5. Curvas de Resposta

As curvas de resposta mostram a relação entre investimento e receita incremental para cada canal em regime estacionário. O ponto atual de investimento está marcado na curva — se está na região íngreme, há espaço para crescer; se está no platô, já saturou.

In [8]:
# Curvas de resposta por canal
fig_rc = plot_response_curves(mmm)
fig_rc.show()

## Conclusão

O modelo de Marketing Mix está treinado e validado com diagnósticos sólidos:

- **R² ~0.88** com resíduos bem comportados — o modelo captura a dinâmica da receita.
- O **baseline** (vendas orgânicas + contexto) representa a maior fatia da receita, como esperado.
- **Email Marketing e Conteúdo Orgânico** têm o maior ROI — canais subinvestidos com alto retorno marginal.
- As **curvas de resposta** mostram onde cada canal está em relação ao ponto de saturação.

No próximo notebook (**03 — Otimização de Budget**), usaremos esses coeficientes para encontrar a alocação ótima de investimento entre canais.